## Setup — Connect to the UGOT

Import the UGOT SDK and helper libraries, then:
- **Initialize** the UGOT at its IP address (opens a gRPC connection on port 50051).
- **Load vision models** that will be used later in the notebook.
- **Open the camera** stream so later cells can read live frames.

Available models loaded here:
| Model | Purpose |
|---|---|
| `color_recognition` | Identifies dominant colors in the camera frame |
| `word_recognition` | Reads printed text (OCR) |
| `line_recognition` | Detects lines for line-following tasks |
| `face_recognition` | Identifies registered faces by name |
| `apriltag_qrcode` | Detects AprilTag fiducial markers and QR codes |

In [ ]:
import time
import importlib
import cv2
import numpy as np
from ugot import ugot
import pose_yolo

# Reload the pose control module so any edits to pose_yolo.py take effect
# without restarting the kernel.
importlib.reload(pose_yolo)
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

# Create the robot controller object
got = ugot.UGOT()

# Connect to the UGOT robot using its IP address.
# Change this string if your robot is on a different IP.
got.initialize("192.168.88.1")

# Load the built-in computer vision models that will be used later in the notebook.
# You can load additional models here if needed — just add the model name to the list.
got.load_models(
    [
        "color_recognition",  # detects dominant colors 
        "word_recognition",  # OCR: reads printed text
        "line_recognition",  # for line-following tasks
        "face_recognition",  # identifies registered faces by name
        "apriltag_qrcode" # AprilTag recognition
    ]
)

# Select the default line-tracking mode.
# 0 = single-line mode (follows one line at a time)
got.set_track_recognition_line(0)

# Open the camera stream so later functions can read live frames
got.open_camera()

print("Done!")

## Live Camera Preview

Before running any vision-based behavior, it's useful to check that the camera feed is working and the robot can see what you expect.

The `display_frame()` helper captures a single frame from the robot's camera and displays it inline in the notebook. It handles the full decoding pipeline:
- `read_camera_data()` returns the raw frame as compressed bytes
- `np.frombuffer` + `cv2.imdecode` decode those bytes into a NumPy image array
- `cv2.cvtColor` converts from BGR (OpenCV's default color order) to RGB (what the notebook display expects)

The loop below calls `display_frame()` as fast as possible, with `clear_output(wait=True)` replacing each frame before drawing the next — giving you a live video preview directly inside the notebook.

Press the **Stop** button or interrupt the kernel to exit.

> **Heads up:** Decoding and rendering frames is computationally expensive. Use this cell to verify your camera setup and field of view, then stop it before running any time-sensitive behaviors.

In [ ]:
got.open_camera()


def display_frame():
    frame = got.read_camera_data()
    if frame is not None:  # Checks if the frame is real!
        nparr = np.frombuffer(frame, np.uint8)  # Reads in the image data
        data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)  # Turns numbers into color
        frame_rgb = cv2.cvtColor(data, cv2.COLOR_BGR2RGB)

        display(Image2.fromarray(frame_rgb))
        clear_output(wait=True)

In [ ]:
try:
    while True:
        display_frame()

except KeyboardInterrupt:
    print("Done!")

## Helper Functions

The cells below define all reusable functions used in the **Combined** sequence at the bottom of the notebook. They cover four capabilities:

| Function | Purpose |
|---|---|
| `face_find_and_approach()` | Spin to search for a named person, then drive up to them |
| `AP_centralization_approaching()` | Center on a visible AprilTag and drive toward it until close |
| `pick_up()` | Use the arm to grab the object identified by the nearest AprilTag |

Run all three cells once before executing the Combined sequence.

### `face_find_and_approach` — Find a Person and Walk Up to Them

This function combines face recognition and movement to make the robot:
1. **Spin** in place until it spots the target person (by face or by reading their name tag)
2. **Approach** the person, keeping them centered in frame, until it gets close enough

**How the approach works:**
- The camera frame is 640 px wide, so the center is at x = 320.
- If the face center (`c_x`) is too far **left** of center, the robot strafes left while moving forward.
- If `c_x` is too far **right**, it strafes right while moving forward.
- If the face is roughly centered but the detected face **height** (`h`) is still small (meaning the robot is far away), it moves straight forward.
- Once the face height exceeds the `height` threshold (meaning the robot is close enough), it stops.
- If the face is lost mid-approach, the robot inches forward slowly until it reappears.

**Parameters:**
| Parameter | Default | Description |
|---|---|---|
| `gap` | `10` | Pixel tolerance around the center (x = 320 ± gap). Within this band the robot won't strafe. |
| `target_name` | `"Stranger"` | The name to search for; must match either the face recognition label or text the OCR can read. |
| `turn_spd` | `15` | Speed used when spinning to search for the target. |
| `strafe_spd` | `10` | Sideways speed used to re-center the target in frame during approach. |
| `fwd_spd` | `10` | Forward speed used during approach. |
| `height` | `80` | How tall (in pixels) the face bounding box must be before the robot considers itself close enough to stop. Decrease this to stop further away; increase it to get closer. |
| `adjust_turn` | `10` | How far the robot turns to center itself after first spotting the target, in degree-units. Increase if the robot tends to approach at an angle. |

In [ ]:
def face_find_and_approach(
    gap=10, target_name="Stranger", turn_spd=15, strafe_spd=10, fwd_spd=10, height=80, adjust_turn=10
):
    """
    Spin until the target person is found, then approach them.

    Phase 1 — Search:
        The robot turns continuously, checking each frame for the target's
        face (via face_recognition) or name tag (via OCR). When found, it
        stops and does a small corrective turn to face them, then moves to Phase 2.

    Phase 2 — Approach:
        The robot drives toward the face, strafing left/right to keep it
        centered in frame, until the face appears large enough (close enough).
    """
    face_name = None  # Will hold the name from face recognition once a face is found

    try:
        # Phase 1: Spin and search
        while True:
            # Turn slowly to scan the environment
            got.mecanum_turn_speed(turn=3, speed=turn_spd)

            # Read text visible in the frame (e.g. a name tag)
            name = got.get_words_result()

            # Check for any recognized faces in the frame
            faces = got.get_face_recognition_total_info()
            if faces:
                face_name = faces[0][0]  # faces[0] = first face; [0] = its name

            # If either the OCR text or the face name matches our target, we found them!
            if name == target_name or face_name == target_name:
                got.mecanum_stop()
                print(f"Saw {target_name}!")

                # Small corrective turn to center the robot on the target
                # turn=3 is clockwise; times=10, unit=2 means ~10 degree-units
                got.mecanum_turn_speed_times(turn=3, speed=20, times=adjust_turn, unit=2)
                break  # Exit Phase 1, move on to Phase 2

        # Phase 2: Approach the target
        while True:
            name = got.get_words_result()
            faces = got.get_face_recognition_total_info()

            if not faces:
                # Lost the face; inch forward slowly to try to find it again
                got.mecanum_translate_speed(angle=0, speed=fwd_spd)
            else:
                c_x = faces[0][1]  # Horizontal center of the face in the frame (0–640 px)
                h = faces[0][3]  # Height of the face bounding box (proxy for distance)
                if h < height:
                    if c_x < 320 - gap:
                        # Face is too far LEFT — strafe left while moving forward
                        got.mecanum_move_xyz(
                            x_speed=-strafe_spd, y_speed=fwd_spd, z_speed=0
                        )
                    elif c_x > 320 + gap:
                        # Face is too far RIGHT — strafe right while moving forward
                        got.mecanum_move_xyz(x_speed=strafe_spd, y_speed=fwd_spd, z_speed=0)
                    else:
                        # Face is centered but still small (far away) — move straight forward
                        got.mecanum_move_xyz(x_speed=0, y_speed=fwd_spd, z_speed=0)
                else:
                    # Face is centered AND large enough — we've arrived!
                    got.mecanum_stop()
                    print(f"Reached {name}!")
                    break  # Done

            clear_output(wait=True)

        got.mecanum_stop()

    except KeyboardInterrupt:
        print("Done")
        got.mecanum_stop()

### `AP_centralization_approaching` and `pick_up` — AprilTag Approach and Arm Pickup

These two functions work together to locate a tagged object and retrieve it with the robot's arm.

---

#### `AP_centralization_approaching(distance, gap, fwd_spd, strafe_spd)`

Drives toward a visible AprilTag while keeping it centered in the camera frame, then stops once the robot is within `distance` meters.

The approach logic mirrors `face_find_and_approach`:
- If the tag is **left** of center (x < 320 − gap): strafe left while moving forward
- If the tag is **right** of center (x > 320 + gap): strafe right while moving forward
- If the tag is **centered but too far**: move straight forward
- If the tag is **centered AND close enough** (distance ≤ threshold): stop

| Parameter | Default | Description |
|---|---|---|
| `distance` | `0.15` | Stop when the tag is within this many **meters**. |
| `gap` | `20` | Pixel tolerance around frame center before strafing kicks in. |
| `fwd_spd` | `10` | Forward drive speed. |
| `strafe_spd` | `10` | Left/right correction speed. |

---

#### `pick_up()`

Assumes the robot is already aligned and close (i.e., `AP_centralization_approaching` has just finished). Uses the AprilTag's current pixel position and estimated distance to calculate arm joint angles, then:
1. Moves arm to a neutral ready position and **opens** the gripper
2. Computes `joint1` (base rotation) from the tag's horizontal pixel offset
3. Computes `joint3` (extension) from the tag's estimated distance in meters
4. Extends the arm to the computed pick-up pose
5. **Closes** the gripper and lifts the arm back to the carry position

> **Note:** If the tag disappears between the approach and the pick-up (e.g., because the robot got too close for the camera to see it), an `IndexError` is caught and a warning is printed instead of crashing.

In [ ]:
def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    """
    Drive toward a detected AprilTag, keeping it centered in the camera frame.

    Parameters:
        distance  (float): Stop when the tag is within this many meters (default 0.15 m).
        gap       (int):   Pixel tolerance around center (320 px) before strafing (default 20 px).
        fwd_spd   (int):   Forward drive speed percentage (default 10 cm/s).
        strafe_spd(int):   Left/right correction speed percentage (default 10 cm/s).
    """
    # Get an initial reading to confirm a tag is visible before entering the loop.
    AP_info = got.get_apriltag_total_info()
    AP_x = AP_info[0][1] # Horizontal pixel position of the tag (0=left, 640=right)
    AP_distance = AP_info[0][6] # Estimated distance to the tag in meters

    while True:
        try:
            # Refresh tag data every iteration for responsive corrections.
            AP_info = got.get_apriltag_total_info()
            AP_x = AP_info[0][1]
            AP_distance = AP_info[0][6]

            if AP_x < 320 - gap:
                # Tag is to the LEFT of center — strafe left to re-align.
                # mecanum_move_xyz(x, y, z): x=strafe, y=forward, z=rotation
                got.mecanum_move_xyz(-strafe_spd, strafe_spd, 0)
            elif AP_x > 320 + gap:
                # Tag is to the RIGHT of center — strafe right to re-align.
                got.mecanum_move_xyz(strafe_spd, strafe_spd, 0)
            elif AP_distance > distance:
                # Tag is centered but still too far — drive straight forward.
                got.mecanum_move_xyz(0, fwd_spd, 0)
            else:
                # Tag is centered AND within target distance — stop and exit.
                got.mecanum_stop()
                print("It's too close, let's stop.")
                break
        except IndexError:
            print("AprilTag is too close!")


def pick_up():
    """
    Pick up the object identified by the closest visible AprilTag.

    Assumes the robot is already aligned and close to the target
    (i.e., AP_centralization_approaching() has just completed).
    """
    # Read the tag's current position and distance for arm targeting.
    AP_info = got.get_apriltag_total_info()
    try:
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        # Move arm to a neutral ready position and open the gripper.
        # joint_control(j1, j2, j3, duration_ms): j2=30, j3=30 tilts arm slightly forward.
        got.mechanical_joint_control(0, 30, 30, 1000)
        got.mechanical_clamp_release() # Open gripper before extending arm
        time.sleep(2) # Wait for gripper to fully open

        # Calculate arm joint angles based on the tag's camera position.
        # joint1 (base): convert pixel offset from center to degrees.
        #   Negative factor corrects for the camera being mirrored horizontally.
        joint1 = int((AP_x - 320) * -1 / 10)

        # joint3 (furthest): convert distance (m) to an extension angle.
        # The -70 offset accounts for the arm's resting angle calibration.
        joint3 = int(AP_distance * 100 - 70)

        # Move arm to the computed pick-up pose.
        got.mechanical_joint_control(joint1, 0, joint3, 500)
        print(f"Joint1 value is: {joint1}, Joint3 value is: {joint3}.")
        time.sleep(1) # Wait for arm to reach the target pose

        # Grasp the object and lift the arm back to the carry position.
        got.mechanical_clamp_close()
        time.sleep(2)  # Wait for gripper to fully close before lifting
        got.mechanical_joint_control(0, 30, 30, 1000)  # Return arm to neutral carry pose
    except IndexError:
        print("AprilTag is too close!")

### Live Face & Text Recognition Preview

This cell runs a live camera loop that overlays both OCR text results and face recognition results directly on the frame — handy for verifying that both models are working before using them in the Combined sequence.

- **Green text (top):** Whatever printed text the OCR model can read in the frame
- **Green text (below):** The face data: `[name, center_x, center_y, height]`

Press the **Stop** button or interrupt the kernel to exit.

In [ ]:
# Ensure the correct models are loaded and the camera is open
# !!!This will NOT be needed on the actual competition. It is here for testing purposes only.
got.load_models(["word_recognition", "face_recognition"])
got.open_camera()

try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Overlay any detected text in the top-left corner
        name = got.get_words_result()
        if name:
            cv2.putText(img, f"text: {name}", (20, 40), 0, 0.8, (0, 255, 0), 2)

        # Overlay face recognition data if any faces are detected.
        # Each face entry is: [name, center_x, center_y, height]
        faces = got.get_face_recognition_total_info()
        if faces:
            face = faces[0]  # Only look at the first detected face
            cv2.putText(img, f"face: {face}", (20, 70), 0, 0.8, (0, 255, 0), 2)

        # Encode the annotated frame as JPEG and display it inline in the notebook
        _, jpeg = cv2.imencode(".jpg", img)
        clear_output(wait=True)
        display(Image(data=jpeg.tobytes()))

except KeyboardInterrupt:
    print("Done")
    got.mecanum_stop()

## Combined Recognition

This section chains together the helper functions defined above into a full multi-step autonomous routine. Make sure all helper cells have been run before executing these cells.

> **Before running:** Re-load all models and re-open the camera using the cell below if you have restarted the kernel since Setup.

In [ ]:
# The following is run at the start of the notebook as well, but if you restart the kernel
# then you will need to reload your models and your camera
got.load_models(
    [
        "color_recognition",  # detects dominant colors 
        "word_recognition",  # OCR: reads printed text
        "line_recognition",  # for line-following tasks
        "face_recognition",  # identifies registered faces by name
        "apriltag_qrcode" # AprilTag recognition
    ]
)

got.open_camera()

### Step 1 — Color Sequence, then AprilTag Pickup

This cell runs a two-phase routine:

**Phase 1 — Color sequence:**
The robot watches the camera for colors in a specific order. It must see **Red first**, then **Green**. Detecting Red alone doesn't trigger anything — it simply unlocks Green detection. Once Green is seen after Red, the robot flashes the matching background color on its screen and moves on.

This ordered detection prevents accidental triggers (e.g. the robot won't react to Green if it hasn't already seen Red).

| Screen background value | Color shown |
|---|---|
| `3` | Red |
| `6` | Green |

**Phase 2 — AprilTag approach and pickup:**
After the color sequence completes, the robot looks for an AprilTag. If one is visible, it:
1. Centers on it and approaches to within 17 cm (`AP_centralization_approaching`)
2. Picks up the object (`pick_up`)

If no tag is visible yet, the robot slowly moves forward (`mecanum_move_speed(0, 15)`) until one comes into view.

In [ ]:
red_seen = False  # Flag to track if Red has been detected yet
got.set_track_recognition_line(line_type=0)  # 0 is a single track line
got.screen_display_background(0) # Reset display background
got.mechanical_joint_control(angle1=0, angle2=45, angle3=45, duration=500)
time.sleep(1)
print("Starting color recognition...")

try:
    while True:
        color = got.get_color_total_info()[0]  # Read fresh color data each loop

        if not red_seen:
            if color == "Red":
                got.screen_display_background(3)  # Red background
                print("Red detected!")
                red_seen = True  # Unlock green detection
        else:
            if color == "Green":
                got.screen_display_background(6)  # Green background
                print("Green detected!")
                time.sleep(1)
                break  # Both colors seen — exit loop

    print("Starting AprilTag recognition...")

    while True:
        AP_info = got.get_apriltag_total_info()

        if AP_info:
            AP_centralization_approaching(
                distance=0.14, gap=20, fwd_spd=10, strafe_spd=5
            )
            # Adjust moving forward if necessary
            # got.mecanum_move_speed_times(direction=0, speed=10, times=5, unit=1)
            pick_up()
            print("Picked up!")
            time.sleep(2)
            break
        else:
            got.mecanum_move_speed(0, 15)

    print("Finished!")

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")

### Step 2 — Pose Control, then Find and Approach a Person

This cell chains two behaviors back-to-back:

**Part 1 — Pose Control:**
The robot is driven by the operator's body gestures (via `run_pose_control_inline`) until the EXIT gesture is performed or the function returns naturally. A 2-second pause and a `clear_output()` clean up the display before the next step.

See the `pose_yolo.py` module (and the documented `mar_28` notebook) for a full parameter reference. Key settings here:
- `enable_robot=True` — the robot will move in response to gestures
- `robot_ip="192.168.88.1"` — must match the IP used in Setup
- `got=got` — passes the already-initialized robot object in so the function doesn't open a second connection

**Part 2 — Face Find and Approach:**
After pose control exits, the robot immediately searches for the person named in `TARGET_NAME` and drives up to them using `face_find_and_approach`.

> **Tip:** Change `TARGET_NAME` to match the name registered with `face_recognition_add_name` (or the text on their name tag).

In [ ]:
TARGET_NAME = "Ryan"

try:
    run_pose_control_inline(
        robot_ip="192.168.1.124",
        forward_speed=30,
        backward_speed=30,
        turn_speed=45,
        camera_index=0,
        model_path="yolov8n-pose.pt",
        up_margin_factor=0.6,
        down_margin_factor=0.6,
        min_conf=0.3,
        enable_robot=True,  # <-- Robot is ENABLED: it will now move!
        debounce_frames=2,
        max_frames=None,
        got=got
    )
    time.sleep(2)
    clear_output()

    face_find_and_approach(
    gap=15, target_name=TARGET_NAME, turn_spd=10, strafe_spd=10, fwd_spd=10, height=120, adjust_turn=10
)
    got.mecanum_move_speed_times(direction=1, speed=20, times=15, unit=1) # Go backwards a bit to not hit face
    got.mechanical_joint_control(angle1=0, angle2=-30, angle3=-30, duration=500)
    time.sleep(1)
    got.mechanical_clamp_release()
    time.sleep(1)
    got.mechanical_joint_control(angle1=0, angle2=45, angle3=45, duration=500)

except KeyboardInterrupt:
    print("Done")

In [ ]:
try:
    run_pose_control_inline(
        robot_ip="192.168.88.1",
        forward_speed=15,
        backward_speed=15,
        turn_speed=25,
        camera_index=0,
        model_path="yolov8n-pose.pt",
        up_margin_factor=0.6,
        down_margin_factor=0.6,
        min_conf=0.3,
        enable_robot=True,  # <-- Robot is ENABLED: it will now move!
        debounce_frames=2,
        max_frames=None,
        got=got
    )
except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")